# Vertebral alignment on a sagittal spine study

**Segmenting vertebral bodies without a model, numbering levels from the sacrum, measuring spondylolisthesis in millimetres, and letting a measured slip become the principal finding**

A radiology read of a lumbar spine has to say, level by level, whether the
vertebrae are stacked in line. A vertebra that has slid forward on the one
below (**anterolisthesis**, usually called spondylolisthesis) or backward
(**retrolisthesis**) narrows the canal and the foramina at that level and is
often the main finding of the examination. Looking at a few rendered images
is not a reliable way to catch it: a step of 5–8 mm is a few pixels on a
downsized montage, and a reader asked "what is wrong here" tends to name the
most conspicuous signal change (a disc bulge, a bright annulus) instead.

`ov.radiology.measure_spine_alignment` measures it:

1. segment the vertebral bodies and the sacrum on a sagittal slice (no trained model),
2. order them along the column and **number them from the sacrum upward**,
3. for every disc level, measure the translation of the upper body over the lower one
   (mm in patient coordinates, % of the lower endplate, Meyerding grade), disc height,
   landmarks and a QC flag.

`ov.radiology.ledger_add_measurement` then writes the result into the read
ledger, where `ov.radiology.draft_principal` can promote a measured slip to
the principal finding — mapped onto the caller's own vocabulary.

This notebook builds a synthetic DICOM study with a **known slip**, so every
number can be compared with the truth.

In [1]:
import json
import os
import pathlib
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import omicverse as ov

T0 = time.time()
WORK = pathlib.Path('spine_phantom')
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## 1. Principle

### 1.1 What is measured (Meyerding, Taillard, the posterior line)

On a sagittal image each vertebral body is a box with four corners:
posterior-superior (PS), anterior-superior (AS), posterior-inferior (PI) and
anterior-inferior (AI). For the disc level between an upper body $U$ and a
lower body $L$ (or the sacrum):

$$\text{offset} = (\mathbf{PI}_U - \mathbf{PS}_L)\cdot\hat{\mathbf e}_L,\qquad
\hat{\mathbf e}_L = \frac{\mathbf{AS}_L - \mathbf{PS}_L}{\lVert \mathbf{AS}_L - \mathbf{PS}_L\rVert}$$

i.e. how far the back of the upper body has moved along the lower body's
superior endplate, positive **anteriorly**. Expressed as a percentage of the
endplate length $\lVert \mathbf{AS}_L - \mathbf{PS}_L\rVert$ this is the
**Taillard index**; binned in quarters it is the **Meyerding grade**
(I ≤ 25 %, II ≤ 50 %, III ≤ 75 %, IV ≤ 100 %, V > 100 % = spondyloptosis;
Meyerding 1932; StatPearls *Spondylolisthesis*, NBK430767). The same offset
is what a reader sees as a **step in the posterior vertebral body line**
(George's line): on a normal spine the posterior walls form one smooth curve.

A level is reported **abnormal** when the offset is at least `min_slip_mm`
(default 3 mm, the threshold most epidemiological studies use for
degenerative spondylolisthesis).

### 1.2 Where the corners come from

A corner is a poorly defined point: endplates are rounded, osteophytes grow
there, and partial volume blurs it. Three ways of turning a segmented body
into a slip were compared on phantoms (section 5):

| method | how | weakness |
|---|---|---|
| **edge fit (default)** | fit a line to the middle of each wall and each endplate; the corner is the intersection ("virtual corner") | needs a clean wall |
| corner points | the contour point nearest each corner of the body's minimum-area rectangle | follows rounded corners / spurs |
| centroid | displacement of body centres along the endplate | biased whenever adjacent bodies differ in size or tilt; undefined against the sacrum (NaN) |

The method reported is the edge fit; the corner-point value is returned too,
and when the two disagree by more than `agree_mm` the level's `qc` becomes
`"check"`.

### 1.3 Segmentation without a model

Classical vertebra segmentation on MRI clusters or thresholds intensities
and cleans up with morphology; edges alone are unreliable because the edge
strengths vary widely (see e.g. arXiv:1605.02460). The function:

1. divides the slice by a smoothed local mean (removes receive-coil shading),
2. thresholds it over a grid of **intensity windows** — bright *and*
   intermediate windows, so it works where marrow is bright (T1, CT) and
   where it is intermediate between dark cortex and bright discs / CSF (T2),
3. keeps components that are compact and box-like at vertebral-body size
   (`body_size_mm`, `min_extent`, `min_solidity`, `max_aspect`),
4. picks the **longest smooth chain** of similar boxes stacked along the
   craniocaudal axis (a dynamic programme over candidates; cf. the
   part-based detector + chain model of Lootus et al. 2013),
5. takes the **sacrum** as the elongated bony component directly below the
   lowest body; its top is the S1 segment.

### 1.4 Numbering from the sacrum, and why it is uncertain

Radiologists number lumbar levels on sagittal images from the sacrum upward,
and so does the function (`level_names`, default S1, L5, L4, …). A
lumbosacral transitional vertebra (≈ 1/3 of people) or a sixth lumbar
vertebra make that count wrong (SPIDER dataset paper, arXiv:2306.12217), so
the result carries `label_confidence` and a warning; without an identified
sacrum the levels are relative (`V1` = lowest body found).

## 2. Build a phantom with a known slip

Vertebral bodies are rounded boxes (dark cortex, marrow inside) on a
lordotic curve that starts at the S1 endplate (sacral slope 38°) and bends
back by `lordosis_deg`; the sacrum is a tapered block. Body cross-sections
are elliptical, so a slice off the midline sees a shallower body — slices
are integrated over their thickness. The distractors are the canal, the
posterior elements, subcutaneous fat and bowel-like blobs; acquisition
adds a smooth multiplicative bias field and Rician noise. The truth per
level is computed from the analytic (sharp) corners with the formula of §1.1.

In [2]:

from typing import Dict, List, Optional

import numpy as np

CONTRAST = {
    #            marrow cortex disc  disc_degen csf   soft  fat   bowel
    "T1": dict(marrow=0.70, cortex=0.12, disc=0.30, disc_degen=0.22, csf=0.15, soft=0.38, fat=0.95,
               bowel=(0.2, 0.6)),
    "T2": dict(marrow=0.45, cortex=0.10, disc=0.85, disc_degen=0.25, csf=1.00, soft=0.30, fat=0.80,
               bowel=(0.3, 0.9)),
    "CT": dict(marrow=0.80, cortex=1.00, disc=0.35, disc_degen=0.35, csf=0.30, soft=0.40, fat=0.25,
               bowel=(0.0, 0.45)),
}


def _rot(phi):
    """Unit column axis u (superior-ish) and endplate axis e (anterior) for a tilt
    ``phi`` (radians, positive = column leaning anteriorly going up); 2-D
    vectors in (anterior, superior) coordinates."""
    u = np.array([np.sin(phi), np.cos(phi)])
    e = np.array([np.cos(phi), -np.sin(phi)])
    return u, e


def _rounded_rect(c, u, e, h, d, r, n=6):
    pts = []
    r = min(r, h / 2 - 0.1, d / 2 - 0.1)
    for sx, sy, a0 in ((1, 1, 0), (-1, 1, 90), (-1, -1, 180), (1, -1, 270)):
        cx, cy = sx * (d / 2 - r), sy * (h / 2 - r)
        for a in np.linspace(a0, a0 + 90, n):
            t = np.deg2rad(a)
            x, y = cx + r * np.cos(t), cy + r * np.sin(t)
            pts.append(c + x * e + y * u)
    return np.array(pts)


def spine_geometry(*, n_lumbar: int = 5, body_h: float = 27.0, body_d: float = 38.0,
                   disc_h: float = 10.0, sacral_slope_deg: float = 38.0, lordosis_deg: float = 50.0,
                   slip_level: Optional[int] = 0, slip_mm: float = 0.0, retro: bool = False,
                   sacrum_len: float = 95.0) -> Dict:
    """Analytic geometry (mm, (anterior, superior) coordinates).

    Body ``k`` (k = 1..n_lumbar from the bottom) sits above disc ``k - 1``;
    body 0 is the sacrum's S1 segment.
    """
    total = n_lumbar * (body_h + disc_h)
    kappa = np.deg2rad(lordosis_deg) / total
    phi0 = np.deg2rad(sacral_slope_deg)
    u0, e0 = _rot(phi0)
    ps1 = np.array([0.0, 0.0])                      # S1 posterior-superior corner
    as1 = ps1 + body_d * e0
    sacrum = np.array([ps1, as1, as1 - sacrum_len * u0 - 0.35 * body_d * e0,
                       as1 - sacrum_len * u0 - 0.35 * body_d * e0 - 0.45 * body_d * e0,
                       ps1 - 0.75 * sacrum_len * u0])
    bodies = [{"label": "S1", "PS": ps1, "AS": as1, "u": u0, "e": e0, "kind": "sacrum"}]
    s_arc = 0.0
    top_c = ps1 + body_d / 2 * e0                    # superior endplate centre of the body below
    shift = np.zeros(2)
    for k in range(1, n_lumbar + 1):
        phi_disc = phi0 - kappa * (s_arc + disc_h / 2)
        ud, _ = _rot(phi_disc)
        inf_c = top_c + disc_h * ud
        phi_b = phi0 - kappa * (s_arc + disc_h + body_h / 2)
        u, e = _rot(phi_b)
        c = inf_c + body_h / 2 * u
        if slip_level is not None and k - 1 == slip_level and slip_mm:
            e_low = bodies[-1]["e"]
            shift = shift + (-1 if retro else 1) * slip_mm * e_low
        c = c + shift
        b = {"label": f"V{k}", "c": c, "u": u, "e": e, "kind": "body",
             "PI": c - body_h / 2 * u - body_d / 2 * e, "AI": c - body_h / 2 * u + body_d / 2 * e,
             "PS": c + body_h / 2 * u - body_d / 2 * e, "AS": c + body_h / 2 * u + body_d / 2 * e}
        bodies.append(b)
        top_c = c + body_h / 2 * u - shift          # the curve continues un-shifted
        s_arc += disc_h + body_h
    truth = []
    for k in range(1, len(bodies)):
        lo, up = bodies[k - 1], bodies[k]
        e_l = (lo["AS"] - lo["PS"]) / np.linalg.norm(lo["AS"] - lo["PS"])
        off = float((up["PI"] - lo["PS"]) @ e_l)
        L = float(np.linalg.norm(lo["AS"] - lo["PS"]))
        n_l = np.array([-e_l[1], e_l[0]])
        if n_l @ up["u"] < 0:
            n_l = -n_l
        dh = float(np.mean([(up["PI"] - lo["PS"]) @ n_l, (up["AI"] - lo["PS"]) @ n_l]))
        truth.append({"disc": k - 1, "offset_mm": off, "slip_pct": 100 * off / L, "lower_endplate_mm": L,
                      "disc_height_mm": dh})
    return {"bodies": bodies, "sacrum": sacrum, "truth": truth, "body_h": body_h, "body_d": body_d}


def _inset(poly: np.ndarray, t: float) -> np.ndarray:
    """Convex polygon shrunk by ``t`` mm (each edge moved inward, consecutive edges intersected)."""
    P = np.asarray(poly, float)
    n = len(P)
    ctr = P.mean(0)
    lines = []
    for k in range(n):
        a, b = P[k], P[(k + 1) % n]
        d = (b - a) / np.linalg.norm(b - a)
        nrm = np.array([-d[1], d[0]])
        if (ctr - a) @ nrm < 0:
            nrm = -nrm
        lines.append((a + t * nrm, d))
    out = []
    for k in range(n):
        (p, r), (q, s_) = lines[k - 1], lines[k]
        A = np.array([r, -s_]).T
        out.append(p + np.linalg.solve(A, q - p)[0] * r)
    return np.array(out)


def _poly_mask(poly_px, shape, ss):
    from skimage.draw import polygon
    m = np.zeros((shape[0] * ss, shape[1] * ss), np.float32)
    rr, cc = polygon(poly_px[:, 0] * ss + (ss - 1) / 2, poly_px[:, 1] * ss + (ss - 1) / 2, m.shape)
    m[rr, cc] = 1
    return m.reshape(shape[0], ss, shape[1], ss).mean((1, 3))


def render_slice(geo: Dict, *, contrast: str = "T1", ps: float = 0.6, fov: float = 260.0,
                 x_mm: float = 0.0, thickness: float = 4.0, body_w: float = 46.0,
                 degenerate_disc: Optional[int] = None, ss: int = 2,
                 rng: Optional[np.random.Generator] = None, n_bowel: int = 6) -> np.ndarray:
    """Noise-free sagittal slice (rows = inferior, cols = posterior)."""
    rng = rng or np.random.default_rng(0)
    C = CONTRAST[contrast]
    n = int(round(fov / ps))
    shape = (n, n)
    # place the column: (anterior, superior) -> (row, col); centre of lumbar spine in the FOV
    allp = np.vstack([np.vstack([b["PS"], b["AS"]]) for b in geo["bodies"]] + [geo["sacrum"]])
    ctr = (allp.min(0) + allp.max(0)) / 2 + np.array([0.0, -5.0])
    origin_a, origin_s = ctr[0] + fov * 0.55, ctr[1] + fov / 2   # anterior of FOV left edge, top

    def px(P):
        P = np.atleast_2d(P)
        return np.stack([(origin_s - P[:, 1]) / ps, (origin_a - P[:, 0]) / ps], 1)

    sub = np.linspace(x_mm - thickness / 2, x_mm + thickness / 2, 5) if thickness > 0 else [x_mm]
    out = np.zeros(shape, np.float32)
    rr, cc = np.mgrid[0:n, 0:n]
    a_mm, s_mm = origin_a - cc * ps, origin_s - rr * ps
    bl = list(rng.uniform(0, 1, (n_bowel, 4)))
    for x in sub:
        f = np.sqrt(max(0.0, 1 - (2 * x / body_w) ** 2))
        img = np.zeros(shape, np.float32)
        # body outline: ellipse around the spine, air outside
        ba = (a_mm - (ctr[0] + 15)) / (fov * 0.47)
        bs = (s_mm - ctr[1]) / (fov * 0.75)
        inside = ba ** 2 + bs ** 2 < 1
        img[inside] = C["soft"]
        img[inside & (ba ** 2 + bs ** 2 > 0.80) & (ba < 0)] = C["fat"]     # posterior subcutaneous fat
        for (p, q, r, v) in bl:   # bowel-like blobs anteriorly
            ca, cs = ctr[0] + 60 + 50 * p, ctr[1] - 80 + 160 * q
            m = ((a_mm - ca) / (12 + 10 * r)) ** 2 + ((s_mm - cs) / (10 + 12 * v)) ** 2 < 1
            img[m & inside] = C["bowel"][0] + (C["bowel"][1] - C["bowel"][0]) * v
        bodies = geo["bodies"]
        # canal + posterior elements behind each lumbar body
        for b in bodies[1:]:
            c, u, e = b["c"], b["u"], b["e"]
            h, d = geo["body_h"], geo["body_d"]
            canal = _rounded_rect(c - (d / 2 + 8) * e, u, e, h + 14, 15, 1)
            img = np.maximum(img * (1 - _poly_mask(px(canal), shape, ss)), 0) + C["csf"] * _poly_mask(px(canal), shape, ss)
            spin = _rounded_rect(c - (d / 2 + 38) * e - 4 * u, u, e, 16, 30, 6)
            pm = _poly_mask(px(spin), shape, ss)
            img = img * (1 - pm) + C["cortex"] * pm
            pm2 = _poly_mask(px(_rounded_rect(c - (d / 2 + 38) * e - 4 * u, u, e, 12, 26, 5)), shape, ss)
            img = img * (1 - pm2) + C["marrow"] * pm2
        # discs
        for k in range(1, len(bodies)):
            lo, up = bodies[k - 1], bodies[k]
            mid_lo = (lo["PS"] + lo["AS"]) / 2
            mid_up = (up["PI"] + up["AI"]) / 2
            poly = np.array([mid_up + f * (up["PI"] - mid_up), mid_up + f * (up["AI"] - mid_up),
                             mid_lo + f * (lo["AS"] - mid_lo), mid_lo + f * (lo["PS"] - mid_lo)])
            m = _poly_mask(px(poly), shape, ss)
            v = C["disc_degen"] if degenerate_disc == k - 1 else C["disc"]
            img = img * (1 - m) + v * m
        # sacrum (cortex + marrow, faint S1-S2 remnant)
        sac = geo["sacrum"]
        sc = sac.mean(0)
        sac_f = sc + (sac - sc) * np.array([f, 1.0])
        m = _poly_mask(px(sac_f), shape, ss)
        img = img * (1 - m) + C["cortex"] * m
        inner = _inset(sac_f, 1.2)
        m = _poly_mask(px(inner), shape, ss)
        img = img * (1 - m) + C["marrow"] * m
        # vertebral bodies: elliptical cross-section -> AP depth scales with f
        for b in bodies[1:]:
            d = geo["body_d"] * f
            if d < 4:
                continue
            outer = _rounded_rect(b["c"], b["u"], b["e"], geo["body_h"], d, 3.0)
            m = _poly_mask(px(outer), shape, ss)
            img = img * (1 - m) + C["cortex"] * m
            innr = _rounded_rect(b["c"], b["u"], b["e"], geo["body_h"] - 2.4, d - 2.4, 2.0)
            m = _poly_mask(px(innr), shape, ss)
            img = img * (1 - m) + C["marrow"] * m
        out += img / len(sub)
    return out


def bias_field(shape, strength: float = 1.0, rng=None) -> np.ndarray:
    """exp(strength * smooth field): strength = log of the max/min ratio (about)."""
    rng = rng or np.random.default_rng(1)
    n0, n1 = shape
    r, c = np.mgrid[0:n0, 0:n1]
    th = rng.uniform(0, 2 * np.pi)
    lin = ((r / n0 - 0.5) * np.cos(th) + (c / n1 - 0.5) * np.sin(th))
    br, bc = rng.uniform(0.2, 0.8, 2)
    bump = np.exp(-(((r / n0 - br) ** 2 + (c / n1 - bc) ** 2) / 0.08))
    f = lin + 0.5 * bump
    f = (f - f.min()) / (np.ptp(f) + 1e-9) - 0.5
    return np.exp(strength * f)


def acquire(img: np.ndarray, *, noise: float = 0.03, bias: float = 0.0, rng=None, scale: float = 1000.0):
    rng = rng or np.random.default_rng(2)
    x = img * (bias_field(img.shape, bias, rng) if bias else 1.0)
    if noise:
        s = noise * float(np.percentile(img[img > 0], 90))
        x = np.sqrt((x + rng.normal(0, s, x.shape)) ** 2 + rng.normal(0, s, x.shape) ** 2)
    return np.clip(x * scale, 0, 65000)


def write_sag_series(root, desc, slices: List[np.ndarray], *, ps: float, spacing: float, x0: float = 0.0,
                     iop=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0), origin=(0.0, -130.0, 130.0), number: int = 1,
                     thickness: Optional[float] = None, modality: str = "MR"):
    """Write slices (lateral positions x0 + k * spacing, k centred) as a DICOM series."""
    from pydicom.dataset import Dataset, FileMetaDataset
    from pydicom.uid import ExplicitVRLittleEndian, generate_uid
    iop = np.asarray(iop, float)
    normal = np.cross(iop[:3], iop[3:])
    suid = generate_uid()
    d = root / desc.replace(" ", "_")
    d.mkdir(parents=True)
    n = len(slices)
    for k, px in enumerate(slices):
        ds = Dataset()
        ds.file_meta = FileMetaDataset()
        ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
        ds.file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.4"
        sop = generate_uid()
        ds.file_meta.MediaStorageSOPInstanceUID = sop
        ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.4"
        ds.SOPInstanceUID = sop
        ds.SeriesInstanceUID = suid
        ds.StudyInstanceUID = "1.2.3.4"
        ds.SeriesDescription = desc
        ds.SeriesNumber = number
        ds.InstanceNumber = k + 1
        ds.Modality = modality
        ds.ImageOrientationPatient = [float(v) for v in iop]
        pos = np.asarray(origin, float) + (x0 + (k - (n - 1) / 2) * spacing) * normal
        ds.ImagePositionPatient = [float(v) for v in pos]
        ds.PixelSpacing = [ps, ps]
        if thickness:
            ds.SliceThickness = float(thickness)
        ds.Rows, ds.Columns = px.shape
        ds.SamplesPerPixel = 1
        ds.PhotometricInterpretation = "MONOCHROME2"
        ds.BitsAllocated = ds.BitsStored = 16
        ds.HighBit = 15
        ds.PixelRepresentation = 0
        ds.PixelData = np.asarray(px).astype(np.uint16).tobytes()
        ds.is_little_endian, ds.is_implicit_VR = True, False
        ds.save_as(str(d / f"IM{k:03d}"), write_like_original=False)
    return suid


def make_study(root, *, contrast: str = "T1", slip_level: Optional[int] = 0, slip_mm: float = 0.0,
               retro: bool = False, noise: float = 0.03, bias: float = 0.0, thickness: float = 4.0,
               lordosis_deg: float = 50.0, sacral_slope_deg: float = 38.0, n_slices: int = 5,
               spacing: float = 4.0, ps: float = 0.6, x_centre: float = 0.0, seed: int = 0,
               desc: Optional[str] = None, degenerate_disc: Optional[int] = None, n_bowel: int = 6,
               iop=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0)):
    """Write a phantom sagittal series; returns (series_dir_root, geometry)."""
    geo = spine_geometry(slip_level=slip_level, slip_mm=slip_mm, retro=retro, lordosis_deg=lordosis_deg,
                         sacral_slope_deg=sacral_slope_deg)
    rng = np.random.default_rng(seed)
    sl = []
    for k in range(n_slices):
        x = x_centre + (k - (n_slices - 1) / 2) * spacing
        im = render_slice(geo, contrast=contrast, ps=ps, x_mm=x, thickness=thickness, rng=np.random.default_rng(seed),
                          degenerate_disc=degenerate_disc, n_bowel=n_bowel)
        sl.append(acquire(im, noise=noise, bias=bias, rng=rng))
    write_sag_series(root, desc or f"Sag {contrast}", sl, ps=ps, spacing=spacing, thickness=thickness, iop=iop,
                     modality="CT" if contrast == "CT" else "MR")
    return geo


In [3]:
# L5 slipped 8 mm forward on S1 (Meyerding I); T1-weighted; oblique sagittal slices
t = np.deg2rad(8)
iop = (0.0, np.cos(t), np.sin(t), 0.0, np.sin(t), -np.cos(t))
geo = make_study(WORK / 'case1', contrast='T1', slip_level=0, slip_mm=8.0, noise=0.05, bias=1.0, iop=iop)
st = ov.radiology.read_dicom_study(WORK / 'case1')
display(st.table())
truth = pd.DataFrame(geo['truth'])
truth.insert(0, 'level', ['L5-S1', 'L4-L5', 'L3-L4', 'L2-L3', 'L1-L2'])
truth.round(2)

                                 series_instance_uid  series_number description plane weighting  is_localizer  n_instances  rows  cols pixel_spacing_mm  slice_spacing_mm            normal laterality
0  1.2.826.0.1.3680043.8.498.50226139580870207705...              1      Sag T1   SAG        T1         False            5   433   433        0.6 x 0.6               4.0  [-1.0, 0.0, 0.0]           

   level  disc  offset_mm  slip_pct  lower_endplate_mm  disc_height_mm
0  L5-S1     0       7.88     20.74               38.0           10.00
1  L4-L5     1      -0.58     -1.53               38.0            9.96
2  L3-L4     2      -0.58     -1.53               38.0            9.96
3  L2-L3     3      -0.58     -1.53               38.0            9.96
4  L1-L2     4      -0.58     -1.53               38.0            9.96

The unslipped levels have a truth offset of about −0.6 mm, not 0: on a
curved column the bodies are rotated relative to each other, so the
posterior corners do not line up exactly. That is anatomy, not slip — which
is why a threshold (`min_slip_mm`) is needed.

## 3. Measure

With a study, the function takes the sagittal diagnostic series (T1 first)
and, without `instance`, segments the middle half of the slices and measures
the one with the most bodies + sacrum and the best-shaped, deepest bodies
(the mid-sagittal slice).

In [4]:
a = ov.radiology.measure_spine_alignment(st)
print('slice', a['instance'], '| sacrum found:', a['sacrum_found'], '| labels:', a['label_confidence'])
for w in a['warnings']:
    print('warning:', w)
cols = ['level', 'offset_mm', 'slip_pct', 'meyerding_grade', 'direction', 'disc_height_mm',
        'offset_corner_mm', 'offset_centroid_mm', 'abnormal', 'qc', 'n_slices_agree']
a['table'][cols]

slice 3 | sacrum found: True | labels: high


   level  offset_mm  slip_pct  meyerding_grade direction  disc_height_mm  offset_corner_mm  offset_centroid_mm  abnormal  qc  n_slices_agree
0  L5-S1        7.8      22.1                1    antero            12.3               7.8                 NaN      True  ok               2
1  L4-L5       -0.9      -2.5                0      none            12.2              -1.0                -3.1     False  ok               0
2  L3-L4       -1.0      -2.8                0      none            12.1              -1.0                -3.1     False  ok               0
3  L2-L3       -0.3      -0.8                0      none            12.4              -0.4                -3.1     False  ok               0
4  L1-L2       -0.3      -0.8                0      none            12.6              -0.9                -3.4     False  ok               0

In [5]:
cmp = a['table'][['level', 'offset_mm', 'slip_pct', 'disc_height_mm']].merge(
    truth[['level', 'offset_mm', 'slip_pct', 'disc_height_mm']], on='level', suffixes=('', '_truth'))
cmp['offset_error_mm'] = cmp.offset_mm - cmp.offset_mm_truth
cmp.round(2)

   level  offset_mm  slip_pct  disc_height_mm  offset_mm_truth  slip_pct_truth  disc_height_mm_truth  offset_error_mm
0  L5-S1        7.8      22.1            12.3             7.88           20.74                 10.00            -0.08
1  L4-L5       -0.9      -2.5            12.2            -0.58           -1.53                  9.96            -0.32
2  L3-L4       -1.0      -2.8            12.1            -0.58           -1.53                  9.96            -0.42
3  L2-L3       -0.3      -0.8            12.4            -0.58           -1.53                  9.96             0.28
4  L1-L2       -0.3      -0.8            12.6            -0.58           -1.53                  9.96             0.28

The offsets agree with the truth to a few tenths of a millimetre, and only
L5-S1 is abnormal (grade I, anterior). The disc height comes out about
2.4 mm larger than the truth: the segmented body is the **marrow**, inside
the dark cortical endplate (1.2 mm here), and on T1 the cortex and the disc
are both dark, so the boundary between them is not visible. The offset is
unaffected (both bodies shrink alike).

Landmarks are returned in patient millimetres (LPS) and in pixels of the
measured slice; the offset is recomputable from the rounded coordinates.

In [6]:
s = st[a['series_instance_uid']]
img = s.pixels(a['instance'])
fig, ax = plt.subplots(1, 2, figsize=(13, 6.5))
ax[0].imshow(img, cmap='gray')
for lv in a['levels']:
    rc = lv['landmarks_rowcol']
    for k, (r, c) in rc.items():
        ax[0].plot(c, r, 'o', ms=3, color='tab:red' if lv['abnormal'] else 'tab:cyan')
    r, c = rc['lower_PS']
    ax[0].text(c + 8, r, f"{lv['level']}  {lv['offset_mm']:+.1f} mm", color='yellow', fontsize=8)
ax[0].set_title('landmarks on the measured slice (red = abnormal level)')
ax[0].axis('off')
r0, c0 = np.array(a['levels'][0]['landmarks_rowcol']['lower_PS']).astype(int)
ax[1].imshow(img[r0 - 70:r0 + 50, c0 - 90:c0 + 30], cmap='gray', extent=(c0 - 90, c0 + 30, r0 + 50, r0 - 70))
for k, (r, c) in a['levels'][0]['landmarks_rowcol'].items():
    ax[1].plot(c, r, 'o', color='tab:red'); ax[1].text(c + 2, r - 2, k, color='yellow', fontsize=8)
pi, ps_, as_ = (np.array(a['levels'][0]['landmarks_rowcol'][k]) for k in ('upper_PI', 'lower_PS', 'lower_AS'))
ax[1].plot([ps_[1], as_[1]], [ps_[0], as_[0]], '-', color='tab:orange', lw=1)
ax[1].set_title('L5-S1: upper PI vs lower PS along the S1 endplate')
plt.tight_layout(); plt.show()
lm = {k: np.array(v) for k, v in a['levels'][0]['landmarks_xyz_mm'].items()}
e = (lm['lower_AS'] - lm['lower_PS']) / np.linalg.norm(lm['lower_AS'] - lm['lower_PS'])
print('L5-S1 landmarks (LPS mm):', {k: v.tolist() for k, v in lm.items()})
print('offset recomputed from landmarks: %.2f mm' % ((lm['upper_PI'] - lm['lower_PS']) @ e))

L5-S1 landmarks (LPS mm): {'upper_PI': [0.0, 37.2, -6.7], 'upper_AI': [0.0, 9.8, -29.4], 'lower_PS': [0.0, 50.1, -8.3], 'lower_AS': [0.0, 25.6, -33.7]}
offset recomputed from landmarks: 7.80 mm


## 4. From measurement to the principal finding

The failure this addresses: a vision reader hinted at "a step at the lowest
disc level", but its strongest words were about a disc at another level, so
the reads-only ledger drafted a disc finding at the wrong level. Below the
ledger holds three severe reads at L4-L5, so the reads-only draft names
L4-L5 (with no finding type: its rules are written for band tears); then
the alignment measurement is added.

In [7]:
def read_entry(xyz, text, severity=3):
    return {'sheet': 'sheet_1', 'series': s.description, 'series_instance_uid': s.uid, 'plane': 'SAG',
            'weighting': s.weighting, 'instance': a['instance'], 'cell': 'B2', 'xyz_mm': list(map(float, xyz)),
            'text': text, 'severity': severity, 'confidence': 'high', 'normal': False, 'structure': None,
            'sheet_note': 'Central structure: L4-L5', 'abnormality': None, 'acute': None,
            'source': 'structure'}

vocabulary = {'structure': ['L3-L4', 'L4-L5', 'L5-S1'],
              'finding_type': ['disc_herniation', 'spondylolisthesis', 'retrolisthesis', 'full_thickness_tear'],
              'acuity': ['acute', 'chronic', 'indeterminate'],
              'grade': ['grade_I', 'grade_II', 'grade_III']}
ledger = str(WORK / 'ledger.json')
l45 = np.array(a['levels'][1]['xyz_mm'])
json.dump([read_entry(l45 + [0, 0, d], 'disc extruded, torn annulus, absent posterior margin') for d in (0, 1, 2)],
          open(ledger, 'w'))
before = ov.radiology.draft_principal(st, ledger, vocabulary)
print('reads only     :', {k: before[k] for k in ('structure', 'finding_type', 'acuity')})

new = ov.radiology.ledger_add_measurement(ledger, st, a)
print(len(new), 'ledger entries added; abnormal:', [e['level'] for e in new if not e['normal']])
print(new[0]['text'])
d = ov.radiology.draft_principal(st, ledger, vocabulary)
print('with measurement:', {k: d[k] for k in ('structure', 'finding_type', 'grade', 'acuity', 'level')})
print('location (mm):', d['location_xyz_mm'], '| evidence:', d['evidence'][0]['sop_instance_uid'][-12:])
print('reads-only draft kept in basis:', d['basis']['reads_only_draft'])

reads only     : {'structure': 'L4-L5', 'finding_type': None, 'acuity': 'indeterminate'}
5 ledger entries added; abnormal: ['L5-S1']
L5-S1: L5 displaced 7.8 mm anteriorly relative to S1 (22.1% of the S1 superior endplate, Meyerding grade I); disc height 12.3 mm; measured, qc ok
with measurement: {'structure': 'L5-S1', 'finding_type': 'spondylolisthesis', 'grade': 'grade_I', 'acuity': 'indeterminate', 'level': 'L5-S1'}
location (mm): [0.0, 30.7, -19.5] | evidence: 457981352439
reads-only draft kept in basis: {'found': True, 'structure': 'L4-L5', 'finding_type': None, 'acuity': 'indeterminate', 'location_xyz_mm': [0.0, 4.3, 7.0]}


Rules (all in `draft_principal`'s docstring):

* only levels whose Meyerding grade ≥ `min_grade` **and** whose QC is `ok` qualify; the largest slip wins;
* the finding type is matched **by name** in the caller's vocabulary
  (`anterolisthesis` / `spondylolisthesis` / `listhesis` / `slip` / … for
  forward, `retrolisthesis` / … for backward), the structure is the closest
  vocabulary term to the level name, a grade term is looked up in any
  vocabulary list whose key mentions "grade" — nothing is hard-coded;
* if the vocabulary has no alignment term the reads-only draft is returned
  (with the reason in `basis['alignment']`);
* `alignment='corroborated'` additionally requires a reader's read near the
  level describing a step / slip; `alignment='off'` ignores measurements;
  a ledger without measurements gives exactly the old draft.

In [8]:
print('off           :', ov.radiology.draft_principal(st, ledger, vocabulary, alignment='off')['finding_type'])
print('corroborated  :', ov.radiology.draft_principal(st, ledger, vocabulary, alignment='corroborated').get('basis', {}).get('principal_source'))
rows = json.load(open(ledger)) + [read_entry(a['levels'][0]['xyz_mm'], 'step-off of the lowest lumbar body on the sacrum', 1)]
print('corroborated + step read:', ov.radiology.draft_principal(st, rows, vocabulary, alignment='corroborated')['structure'])
no_term = dict(vocabulary, finding_type=['disc_herniation', 'full_thickness_tear'])
d2 = ov.radiology.draft_principal(st, ledger, no_term)
print('vocabulary without a slip term ->', d2['finding_type'], '|', d2['basis']['alignment']['reason'])

off           : None
corroborated  : None
corroborated + step read: L5-S1
vocabulary without a slip term -> None | the vocabulary has no alignment / slip finding type


## 5. Robustness: contrast, slice thickness, shading, noise, lordosis, grade

A small sweep (the full comparison in the pull request used 60 random
configurations). Each row is one phantom; the error is at the slipped level
for the three measurement methods of §1.2.

In [9]:
configs = [
    dict(contrast='T1', slip_level=0, slip_mm=0.0, noise=0.03, thickness=3.0, bias=0.0, lordosis_deg=50.0),
    dict(contrast='T1', slip_level=0, slip_mm=5.7, noise=0.08, thickness=6.0, bias=1.2, lordosis_deg=60.0),
    dict(contrast='T2', slip_level=1, slip_mm=6.0, retro=True, noise=0.05, thickness=6.0, bias=1.2, lordosis_deg=50.0),
    dict(contrast='T2', slip_level=1, slip_mm=13.3, noise=0.05, thickness=3.0, bias=0.0, lordosis_deg=30.0),
    dict(contrast='CT', slip_level=0, slip_mm=15.0, noise=0.03, thickness=3.0, bias=0.0, lordosis_deg=30.0),
    dict(contrast='T1', slip_level=2, slip_mm=23.0, noise=0.08, thickness=3.0, bias=1.2, lordosis_deg=60.0),
]
rows = []
for k, cfg in enumerate(configs):
    root = WORK / f'sweep{k}'
    g = make_study(root, seed=k, **cfg)
    r = ov.radiology.measure_spine_alignment(ov.radiology.read_dicom_study(root))
    j = cfg['slip_level']
    tr, lv = g['truth'][j], r['levels'][j]
    rows.append({'contrast': cfg['contrast'], 'noise': cfg['noise'], 'thick_mm': cfg['thickness'], 'bias': cfg['bias'],
                 'lordosis': cfg['lordosis_deg'], 'level': lv['level'], 'truth_mm': round(tr['offset_mm'], 1),
                 'edge_err': lv['offset_mm'] - tr['offset_mm'], 'corner_err': lv['offset_corner_mm'] - tr['offset_mm'],
                 'centroid_err': (lv['offset_centroid_mm'] - tr['offset_mm']) if lv['offset_centroid_mm'] is not None else np.nan, 'grade': lv['meyerding_grade'],
                 'grade_truth': 0 if abs(tr['offset_mm']) < 3 else int(np.ceil(abs(tr['slip_pct']) / 25)),
                 'abnormal_levels': [x['level'] for x in r['levels'] if x['abnormal']], 'labels': r['label_confidence']})
pd.DataFrame(rows).round(2)

  contrast  noise  thick_mm  bias  lordosis  level  truth_mm  edge_err  corner_err  centroid_err  grade  grade_truth abnormal_levels labels
0       T1   0.03       3.0   0.0      50.0  L5-S1      -0.1     -0.18        0.22           NaN      0            0              []   high
1       T1   0.08       6.0   1.2      60.0  L5-S1       5.6      0.12       -0.08           NaN      1            1         [L5-S1]   high
2       T2   0.05       6.0   1.2      50.0  L4-L5      -6.6     -0.32       -0.92         -2.72      1            1         [L4-L5]   high
3       T2   0.05       3.0   0.0      30.0  L4-L5      12.9     -0.28        0.02         -1.08      2            2         [L4-L5]   high
4       CT   0.03       3.0   0.0      30.0  L5-S1      14.9      0.30        0.80           NaN      2            2         [L5-S1]   high
5       T1   0.08       3.0   1.2      60.0  L3-L4      22.4     -0.07       -0.27         -3.27      3            3         [L3-L4]   high

## 6. Pitfalls

* **Level numbering** assumes no transitional vertebra. Check `label_confidence`
  and, when the level matters, count from C2 on a whole-spine localizer.
* **Which slice**: the slip is measured on one slice (the most complete
  column with the deepest bodies). A slice off the midline sees shallower
  bodies; `n_slices_agree` tells how many neighbouring slices show the same
  abnormality.
* **Disc height** is between marrow edges (cortical endplate excluded), so
  it reads 1–3 mm high; compare it between levels, not with external normal values.
* **Supine imaging** under-represents slips that reduce lying down; MRI
  grades can be lower than standing radiographs.
* **Synthetic validation only**: the segmentation thresholds are generic
  shape limits; on real studies, inspect the landmark overlay (section 3)
  before trusting a number, and treat `qc == "check"` as unmeasured.

## References

* Meyerding HW. Spondylolisthesis. *Surg Gynecol Obstet* 1932;54:371–377.
* Taillard W. Le spondylolisthésis chez l'enfant et l'adolescent. *Acta Orthop Scand* 1954;24:115–144.
* StatPearls: Spondylolisthesis (NBK430767); Radiopaedia: Spondylolisthesis.
* Lootus M, Kadir T, Zisserman A. Vertebrae detection and labelling in lumbar MR images. MICCAI-CSI 2013.
* van der Graaf JW et al. Lumbar spine segmentation in MR images: a dataset and a public benchmark (arXiv:2306.12217).
* Wang YXJ et al. Lumbar degenerative spondylolisthesis epidemiology: a systematic review (arXiv:1610.06608).
* Fuzzy c-means vertebra segmentation in MR (arXiv:1605.02460).

In [10]:
print(f'total run time {time.time() - T0:.0f} s')
shutil.rmtree(WORK, ignore_errors=True)

total run time 106 s
